# 02 QLoRA 학습 (HyperCLOVA X SEED 14B Think)

| 항목 | 값 |
|---|---|
| GPU | A100 40GB 또는 80GB (둘 다 가능) |
| 예상 시간 | 약 3~3.5시간 (다운로드 15분, 학습률 2개 × 2 epoch 학습 2.5시간, 검증 생성 4회 30분) |
| 예상 컴퓨트 | 약 40~45 CU (A100 약 13 CU/시간 가정. Colab 리소스 창에서 실제 소모율을 확인해 기록) |
| 디스크 | 학생 원본 float32 59GB |
| 이어 하기 | Trainer 체크포인트(epoch마다)를 Drive `lora/`에 저장하고, 다시 실행하면 마지막 체크포인트에서 이어 한다. 검증 생성도 이어 쓴다. |

사전 등록: `docs/spec.md` §6.1~6.3, `docs/decisions.md` 2026-11-10. 설정을 바꾸면 사전 등록 위반이니 멈추고 기록한다.
준비: repo에서 `scripts/pack_distill_bundle.sh`로 만든 `tomato_distill_bundle.tar.gz`를 Drive `MyDrive/tomato-distill/`에 올린다.

In [ ]:
# 공통 준비: Drive 연결, 묶음 풀기, 공용 코드 import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time, gc
DIR = "/content/drive/MyDrive/tomato-distill"
os.environ["TOMATO_DISTILL_DIR"] = DIR
if not os.path.exists(f"{DIR}/bundle/notebooks/distill_common.py"):
    os.makedirs(f"{DIR}/bundle", exist_ok=True)
    !tar xzf {DIR}/tomato_distill_bundle.tar.gz -C {DIR}/bundle
sys.path.insert(0, f"{DIR}/bundle/notebooks")
import distill_common as dc
prompt = dc.setup()
from tomato.explain import grounding
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U "transformers>=5.9.0" peft bitsandbytes accelerate
import torch, transformers, peft, bitsandbytes
print(transformers.__version__, peft.__version__, bitsandbytes.__version__)

## 데이터: 선생 출력 중 근거 검사·형식 통과분 (01에서 만든 `*_final.jsonl`)

In [ ]:
train_in, val_in = dc.subsets()
inputs = {r["id"]: r["input"] for r in train_in + val_in}
train_t = dc.read_jsonl(dc.DRIVE / "teacher" / "train_final.jsonl")
val_t = dc.read_jsonl(dc.DRIVE / "teacher" / "val_final.jsonl")
print(len(train_t), len(val_t))
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV)
tok.padding_side = "right"
train_ds = [dc.sft_features(tok, prompt, inputs[r["id"]], r["text"]) for r in train_t]
val_ds = [dc.sft_features(tok, prompt, inputs[r["id"]], r["text"]) for r in val_t]
print("최대 토큰", max(len(x["input_ids"]) for x in train_ds))
print(tok.decode(train_ds[0]["input_ids"])[-600:])

## 학습: r 16, alpha 32, 학습률 {1e-4, 2e-4} × epoch {1, 2} (epoch마다 체크포인트)

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import glob

def load_base():
    m = AutoModelForCausalLM.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV, quantization_config=dc.bnb_config(),
                                             dtype=torch.bfloat16, device_map={"": 0})
    return prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)

G = dc.LORA_GRID
for lr in G["lr"]:
    out = dc.DRIVE / "lora" / f"r{G['r']}_lr{lr:g}"
    if (out / "done").exists():
        continue
    model = get_peft_model(load_base(), LoraConfig(r=G["r"], lora_alpha=G["alpha"], lora_dropout=G["dropout"],
                                                   target_modules=G["targets"], task_type="CAUSAL_LM"))
    args = TrainingArguments(output_dir=str(out), num_train_epochs=G["epochs"], learning_rate=lr,
                             per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=8,
                             lr_scheduler_type="cosine", warmup_ratio=0.03, bf16=True, logging_steps=10,
                             eval_strategy="epoch", save_strategy="epoch", save_total_limit=None, seed=dc.SEED,
                             gradient_checkpointing=True, optim="paged_adamw_8bit", report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
    ckpts = sorted(glob.glob(f"{out}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
    (out / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
    (out / "done").write_text("ok")
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

## 설정 선택: 검증(2022) 200개 greedy 생성의 근거 검사 통과율(a·b·c), 동률이면 검증 손실이 낮은 쪽, 그다음 작은 학습률·적은 epoch

In [ ]:
from peft import PeftModel
results = []
for lr in G["lr"]:
    out = dc.DRIVE / "lora" / f"r{G['r']}_lr{lr:g}"
    hist = json.loads((out / "log_history.json").read_text())
    evals = [h for h in hist if "eval_loss" in h]
    ckpts = sorted(glob.glob(f"{out}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    for ep, ck in enumerate(ckpts, start=1):
        path = out / f"val_gen_ep{ep}.jsonl"
        if len(dc.read_jsonl(path)) < len(val_in):
            base, tok_g = dc.load_student(deploy=False)
            model = PeftModel.from_pretrained(base, ck).eval()
            dc.student_generate(model, tok_g, prompt, val_in, batch_size=16, out_path=path, label=f"lr{lr:g} ep{ep}")
            del model, base; gc.collect(); torch.cuda.empty_cache()
        gens = {r["id"]: r["text"] for r in dc.read_jsonl(path)}
        checks = [grounding.check(gens[r["id"]], r["input"]) for r in val_in]
        results.append({"lr": lr, "epoch": ep, "checkpoint": ck, "val_pass": sum(c["pass"] for c in checks) / len(checks),
                        "val_format": sum(c["format"] for c in checks) / len(checks),
                        "val_loss": evals[ep - 1]["eval_loss"]})
chosen = sorted(results, key=lambda x: (-x["val_pass"], x["val_loss"], x["lr"], x["epoch"]))[0]
sel = {"grid": {k: v for k, v in G.items()}, "results": results, "chosen": chosen, "n_train": len(train_ds), "n_val": len(val_in)}
(dc.DRIVE / "lora" / "selection.json").write_text(json.dumps(sel, indent=1, default=str))
print(json.dumps(sel, indent=1, default=str))

In [ ]:
# 고른 어댑터만 lora/chosen/에 복사하고 해시를 남긴다
import shutil
dst = dc.DRIVE / "lora" / "chosen"
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(chosen["checkpoint"], dst, ignore=shutil.ignore_patterns("optimizer.pt", "scheduler.pt", "rng_state*", "trainer_state.json", "training_args.bin"))
print(dc.dir_hashes(dst))

## repo로 가져올 파일
- `lora/selection.json`, 각 설정의 `log_history.json`, `val_gen_ep*.jsonl` → `distill/lora/`
- `lora/chosen/`(adapter_config.json, adapter_model.safetensors) → `distill/lora/chosen/`
  - r 16, 모든 linear 대상이면 adapter 파라미터가 약 6,100만 개라 파일이 약 120MB(bf16)~250MB(fp32)다. GitHub의 파일당 100MB 제한을 넘으므로 push할 때는 Git LFS를 쓰거나 Drive에 두고 해시만 기록한다.